In [2]:
import pandas as pd 
import numpy as np
import joblib
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier

### Helper function

In [3]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

## Load augmented datasets 

In [4]:
train = pd.read_csv('../datasets/augment/train_tvae_9600.csv')
test = pd.read_csv('../datasets/test_shap_66.csv')

X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']
y_test = pd.Series(y_test)
y_train = pd.Series(y_train)

## XGBoost

In [16]:
xgb_params = {
    'tree_method': 'hist',         
    'predictor': 'gpu_predictor',      
    'max_depth': 15,
    'n_estimators': 5000,
    'learning_rate': 0.3, 
    'eval_metric': 'mlogloss',
    'use_label_encoder': False,
    'objective':"multi:softmax", 
    'num_class':len(y_train.unique()),
    'booster': 'gbtree',
    'random_state': 42
}


print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
print("XGBClassifier Finished")

xgb_acc = accuracy_score(y_true=y_test, y_pred=xgb_prediction)
xgb_precision = precision_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_f1 = f1_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_recall = recall_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_cm = confusion_matrix(y_true=y_test, y_pred=xgb_prediction)
xgb_fp = xgb_cm[0, 1]
print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost FP", xgb_fp)
print("XGBoost CM:", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')

XGBClassifier Starting


/Users/trungnguye2n.dev/foami-plus/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [23:32:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.10995984077453613
XGBoost Accuracy: 0.8500986193293886
XGBoost Precision: 0.8450007004445821
XGBoost F1: 0.844784356243763
XGBoost Recall: 0.8500986193293886
XGBoost FP 0
XGBoost CM: [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  35   1   2   0   4  67  35   8  16   0]
 [  0   0   0   0   0   0   0  42 127   0   0   0]
 [  0   0  10   0   1   0   0   5   0  94  59   0]
 [  0   0   0   0   0   0   0   0   0  84  85   0]
 [  0   0   0   0   1   0   0   0   0   0   0 168]]
XGBoost Macro-average TPR: 0.8500986193293886
XGBoost Macro-average FPR: 0.013627398242782857


In [18]:
import os
os.makedirs('./models', exist_ok=True)
joblib.dump(xgb_model, './models/framework_xgb_TVAE.pkl')

['./models/framework_xgb_TVAE.pkl']

In [20]:
# --- Load models ---
xgb_model = XGBClassifier()
xgb_model = joblib.load('./models/framework_xgb_TVAE.pkl')

xgb_preds = xgb_model.predict_proba(X_test)

xgb_prediction = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_true=y_test, y_pred=xgb_prediction)

print("XGBoost Accuracy:", xgb_acc)

XGBoost Accuracy: 0.8500986193293886


## ExtraTree

In [21]:
et_params = {
    "n_estimators": 73,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.013839960098266602
ExtraTrees Accuracy: 0.8574950690335306
ExtraTrees Precision: 0.8569249898577947
ExtraTrees F1: 0.8539873750555161
ExtraTrees Recall: 0.8574950690335306
ExtraTrees FP: 0
ExtraTrees CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  30   0   0   0   0  87  26  10  15   0]
 [  0   0   0   0   0   0   0  32 137   0   0   0]
 [  0   0  10   0   0   0   0   6   0 100  53   0]
 [  0   0   0   0   0   0   0   0   0 106  63   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
ExtraTrees Macro-average TPR: 0.8574950690335306
ExtraTrees Macro-average FPR: 0.012

In [22]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 60, 90, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            87  0.861440  0.857665      0.861440  0.014827
1            90  0.861440  0.857436      0.861440  0.025806
2            89  0.860454  0.856531      0.860454  0.014953
3            88  0.860454  0.856508      0.860454  0.014519
4            85  0.860454  0.856420      0.860454  0.014949
5            86  0.859961  0.855879      0.859961  0.014961
6            82  0.859961  0.855843      0.859961  0.013259
7            83  0.859467  0.855479      0.859467  0.024343
8            84  0.859467  0.855456      0.859467  0.015136
9            80  0.858974  0.854851      0.858974  0.013165

===== Best ExtraTrees report =====
Best n_estimators: 87
ExtraTrees Time: 0.014826536178588867
ExtraTrees Accuracy: 0.861439842209073
ExtraTrees Precision: 0.860935652063454
ExtraTrees F1: 0.8576652245142792
ExtraTrees Recall: 0

In [23]:
joblib.dump(et_model, './models/framework_et_TVAE.pkl')

['./models/framework_et_TVAE.pkl']

## RandomForest

In [25]:
rf_params = {
    "n_estimators": 69,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.027553081512451172
RandomForest Accuracy: 0.8609467455621301
RandomForest Precision: 0.8618419646883306
RandomForest F1: 0.8580802735500943
RandomForest Recall: 0.8609467455621301
RandomForest FP: 0
RandomForest CM: [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  35   0   2   0   1  91  16  11  12   0]
 [  0   0   0   0   0   0   0  39 130   0   0   0]
 [  0   0  13   0   0   0   0   4   0 102  50   0]
 [  0   0   0   0   0   0   0   0   0  98  71   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
RandomForest Macro-average TPR: 0.8609467455621301
RandomForest M

In [26]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 50, 90, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0            51  0.863412  0.860912      0.863412  0.015595
1            53  0.862426  0.860059      0.862426  0.012744
2            50  0.862426  0.859710      0.862426  0.017349
3            61  0.861933  0.859409      0.861933  0.013582
4            55  0.861933  0.859375      0.861933  0.012937
5            63  0.861933  0.859300      0.861933  0.014780
6            57  0.861933  0.859288      0.861933  0.013319
7            71  0.861933  0.859236      0.861933  0.014335
8            52  0.861933  0.859227      0.861933  0.029088
9            67  0.861933  0.859214      0.861933  0.015098

===== Best RandomForest report =====
Best n_estimators: 51
RandomForest Time: 0.015594959259033203
RandomForest Accuracy: 0.8634122287968442
RandomForest Precision: 0.8647288414648105
RandomForest F1: 0.8609123801653848
Rand

In [28]:
joblib.dump(rf_model, './models/framework_rf_TVAE.pkl')

['./models/framework_rf_TVAE.pkl']

In [30]:
from lightgbm import LGBMClassifier, early_stopping

lgbm_params = {
 "boosting_type": "gbdt",
    "objective": "multiclass",
    "num_class": len(y_train.unique()),
    "metric": "multi_logloss",

    "learning_rate": 0.13,
    "n_estimators": 5000,
    "num_leaves": 192,
    "max_depth": -1,

    "min_child_samples": 48,
    "reg_alpha": 0.15,
    "reg_lambda": 0.8,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,

    "device": "cpu",
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1, 
}

# Train LightGBM Model
print("LightGBM Starting")
lgbm_model = LGBMClassifier(**lgbm_params)
lgbm_model.fit(
    X=X_train, y=y_train, 
    eval_set=[(X_test, y_test)],
    callbacks=[early_stopping(stopping_rounds=30)],
)
lgbm_prediction = lgbm_model.predict(X_test)
print("LightGBM Finished")

lgbm_acc = accuracy_score(y_test, lgbm_prediction)
lgbm_precision = precision_score(y_test, lgbm_prediction, average='macro')
lgbm_f1 = f1_score(y_test, lgbm_prediction, average='macro')
lgbm_recall = recall_score(y_test, lgbm_prediction, average='macro')
lgbm_cm = confusion_matrix(y_test, lgbm_prediction)

lgbm_probs = lgbm_model.predict_proba(X_test)
lgbm_auc = roc_auc_score(y_test, lgbm_probs, multi_class='ovr', average='macro')

print("LightGBM report:")
print("LightGBM Accuracy:", lgbm_acc)
print("LightGBM Precision:", lgbm_precision)
print("LightGBM F1:", lgbm_f1)
print("LightGBM Recall:", lgbm_recall)
print("LightGBM ROC AUC:", lgbm_auc)
print("LightGBM CM:\n", lgbm_cm)
lgbm_tpr, lgbm_fpr = calculate_macro_tpr_fpr(lgbm_cm)
print(f'LightGBM TPR: {lgbm_tpr}')
print(f'LightGBM FPR: {lgbm_fpr}')

LightGBM Starting
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[63]	valid_0's multi_logloss: 0.409281
LightGBM Finished
LightGBM report:
LightGBM Accuracy: 0.8525641025641025
LightGBM Precision: 0.8474059900651335
LightGBM F1: 0.8478808667088309
LightGBM Recall: 0.8525641025641025
LightGBM ROC AUC: 0.9877558718022987
LightGBM CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   1 168   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  29   1   1   1   5  77  38   4  12   0]
 [  0   0   0   0   0   0   0  44 125   0   0   0]
 [  0   2   9   0   0   0   0   6   0  97  55   0]
 [  0   0   0   0   0   0   0   0   0  90  79   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
Lig

In [31]:
joblib.dump(lgbm_model, './models/framework_lgbm_TVAE.pkl')

['./models/framework_lgbm_TVAE.pkl']

In [5]:
from catboost import CatBoostClassifier
import time

cat_params = {
    "iterations": 5000,          # nhiều cây hơn
    "depth": 8,                  # đủ phức tạp mà không quá chậm
    "learning_rate": 0.05,       # học chậm hơn, tổng quát hóa tốt hơn
    "l2_leaf_reg": 3,            # regularization chống overfit
    "random_strength": 1,        # thêm randomness
    "bagging_temperature": 1,    # bootstrap sampling
    "loss_function": "MultiClass",
    "task_type": "CPU",
    "thread_count": -1,
    "random_seed": 42,
    "eval_metric": "TotalF1",
    "od_type": "Iter",
    "od_wait": 100,              # kiên nhẫn hơn với early stopping
    "classes_count": len(y_train.unique()),
}

print("CatBoost Starting")
cat_model = CatBoostClassifier(**cat_params)

# fit model
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=False
)

cat_start_time = time.time()
cat_prediction = cat_model.predict(X_test)
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoost Finished")

cat_acc = accuracy_score(y_true=y_test, y_pred=cat_prediction)
cat_precision = precision_score(y_true=y_test, y_pred=cat_prediction, average="macro")
cat_f1 = f1_score(y_true=y_test, y_pred=cat_prediction, average="macro")
cat_recall = recall_score(y_true=y_test, y_pred=cat_prediction, average="macro")
cat_cm = confusion_matrix(y_true=y_test, y_pred=cat_prediction)
cat_fp = cat_cm[0, 1] if cat_cm.shape[0] > 1 else 0

print("CatBoost report:")
print("CatBoost Time:", cat_time)
print("CatBoost Accuracy:", cat_acc)
print("CatBoost Precision:", cat_precision)
print("CatBoost F1:", cat_f1)
print("CatBoost Recall:", cat_recall)
print("CatBoost FP", cat_fp)
print("CatBoost CM:", cat_cm)

cat_tpr, cat_fpr = calculate_macro_tpr_fpr(cat_cm)
print(f"CatBoost Macro-average TPR: {cat_tpr}")
print(f"CatBoost Macro-average FPR: {cat_fpr}")

CatBoost Starting
CatBoost Finished
CatBoost report:
CatBoost Time: 0.014297723770141602
CatBoost Accuracy: 0.8550295857988166
CatBoost Precision: 0.8543015907673928
CatBoost F1: 0.8521785530701033
CatBoost Recall: 0.8550295857988166
CatBoost FP 0
CatBoost CM: [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 168   0   0   1   0   0   0   0   0]
 [  0   0   0   3 166   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   1   1   0 167   0   0   0   0   0]
 [  0   1  38   3   3   0   4  93  11   6  10   0]
 [  0   0   0   0   0   0   0  46 123   0   0   0]
 [  0   3  11   0   0   0   0   5   0  85  65   0]
 [  0   0   0   0   0   0   0   0   0  82  87   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
CatBoost Macro-average TPR: 0.8550295857988166
CatBoost Macro-average FPR: 0.013179128563743947


In [6]:
joblib.dump(cat_model, './models/framework_cat_TVAE.pkl')

['./models/framework_cat_TVAE.pkl']

## LSTM

In [7]:
# ============================
# LSTM cho Tabular — IEC/ICS preset — KHÔNG focus lớp nào
#  - Stratified split (VAL 15%)
#  - StandardScaler
#  - DataLoader (shuffle, không sampler/boost)
#  - LSTM (2 tầng, BiLSTM) + AttnPooling
#  - Mixup công bằng cho mọi lớp
#  - EMA (float-only) device-safe
#  - Early-stopping theo Macro-F1 (VAL)
#  - Đánh giá: Acc, Macro-F1, per-class metrics
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ----- Device & seed -----
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
assert "Label" in train.columns and "Label" in test.columns, "Thiếu cột Label trong train/test"
X_full = train.drop(columns=["Label"]).values
y_full = train["Label"].astype(int).values
X_test = test.drop(columns=["Label"]).values
y_test = test["Label"].astype(int).values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ===== 3) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 2048
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 4) Tiện ích: chia feature thành "chuỗi" =====
# Ta coi features như một chuỗi gồm S steps, mỗi step có input_size = STEP_DIM
# (pad 0 nếu F không chia hết)
STEP_DIM = 16  # bạn có thể thử 8/16/32
def as_sequence(x_np: np.ndarray, step_dim: int = STEP_DIM):
    F = x_np.shape[1]
    S = int(np.ceil(F / step_dim))
    pad = S * step_dim - F
    if pad > 0:
        x_np = np.pad(x_np, ((0,0),(0,pad)), mode="constant")
    return x_np.reshape(-1, S, step_dim), S

# ===== 5) Mô hình LSTM + AttnPooling =====
class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.proj = nn.Linear(d, 1)
    def forward(self, H):        # H: [B,S,D]
        score = self.proj(H).squeeze(-1)           # [B,S]
        w = torch.softmax(score, dim=1)            # [B,S]
        pooled = (H * w.unsqueeze(-1)).sum(1)      # [B,D]
        return pooled

class LSTMTabular(nn.Module):
    def __init__(self, step_dim, hidden=256, layers=2, n_classes=10, dropout=0.2, bidir=True):
        super().__init__()
        self.step_dim = step_dim
        self.in_norm  = nn.LayerNorm(step_dim)
        self.lstm = nn.LSTM(
            input_size=step_dim,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
            bidirectional=bidir
        )
        d_out = hidden * (2 if bidir else 1)
        self.pool = AttnPool(d_out)
        self.head = nn.Sequential(
            nn.Linear(d_out, d_out//2), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(d_out//2, n_classes)
        )
    def forward(self, x):                       # x: [B, F]
        B, F = x.shape
        S = int(math.ceil(F / self.step_dim))
        pad = S * self.step_dim - F
        if pad > 0:
            x = torch.nn.functional.pad(x, (0, pad), value=0.0)
        x = x.view(B, S, self.step_dim)         # [B,S,step_dim]
        x = self.in_norm(x)
        H, _ = self.lstm(x)                     # [B,S,D]
        z = self.pool(H)                        # [B,D]
        return self.head(z)                     # [B,C]

model = LSTMTabular(step_dim=STEP_DIM, hidden=256, layers=2, n_classes=num_classes, dropout=0.15, bidir=True).to(device)

# ===== 6) Loss/optim/scheduler + mixup (công bằng) =====
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

total_epochs = 120
warmup_epochs = 19
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if step < warmup_steps: return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def mixup_pair(x, y, alpha=0.10):
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 7) EMA (float-only) — device-safe =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point(): continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_((decay)).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict(); merged = {}
    for k, v in current.items():
        merged[k] = ema_state.get(k, v).to(device=v.device, dtype=v.dtype)
    model.load_state_dict(merged, strict=True)

# ===== 8) Train + Early-stopping theo Macro-F1 (VAL) =====
best_macro_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_pair(xb, yb, alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step(); scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # Eval VAL với EMA
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    macro_f1 = f1_score(y_val, pred_val, average='macro', zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1, wait = macro_f1, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)
    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_macro_f1:.4f}")
        break

# Nạp EMA tốt nhất
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)
model.eval()

# ===== 9) SUY LUẬN TEST =====
with torch.no_grad():
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

y_pred = logits_test.argmax(1)

# ===== 10) Đánh giá =====
acc = accuracy_score(y_test, y_pred)
macro_f1_all = f1_score(y_test, y_pred, average='macro', zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
print(f"\nLSTM-Tabular — Acc: {acc:.4f} | Macro-F1(all): {macro_f1_all:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

labels_arr = np.arange(num_classes)
prec, rec, f1c, support = precision_recall_fscore_support(
    y_test, y_pred, labels=labels_arr, zero_division=0
)

# FPR per class
fpr_per_class = []
for i in labels_arr:
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    fpr_i = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    fpr_per_class.append(fpr_i)

macro_precision = float(np.mean(prec))
macro_recall    = float(np.mean(rec))
macro_fpr       = float(np.mean(fpr_per_class))

print("\n==== Macro metrics (all classes) ====")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall/TPR: {macro_recall:.4f}")
print(f"Macro-FPR: {macro_fpr:.4f}")

per_class_df = pd.DataFrame({
    "class": labels_arr,
    "precision": prec,
    "recall_TPR": rec,
    "f1": f1c,
    "FPR": fpr_per_class,
    "support": support,
}).round(4)
print("\n==== Per-class metrics (all classes) ====")
print(per_class_df)

: 

In [ ]:
# Save LSTM checkpoint (compatible with LSTMWrapper.from_checkpoint)
import torch


checkpoint = {
    'state_dict': model.state_dict(),
    'scaler': scaler,
    'input_dim': num_features,
    'num_classes': num_classes,
    'step_dim': STEP_DIM,
    'hidden': 256,
    'layers': 2,
    'dropout': 0.15,
    'bidir': True,
}

save_path = './models/framework_lstm_TVAE.pth'
torch.save(checkpoint, save_path)
print(f"LSTM checkpoint saved to {save_path}")
print(f"  input_dim={num_features}, num_classes={num_classes}, step_dim={STEP_DIM}")

## ResDNN


In [ ]:
# ============================
# DNN Tabular (ResNet-style) — ICS preset (EMA-float only) — SAFE DEVICE
# Tập trung lớp 7 & 10 (không đổi hyper-params)
# Tích hợp:
#  - CB-Focal + class weights (Effective Number)
#  - Sampler boost cho lớp 7/10
#  - Mixup giảm khi batch có 7/10
#  - EMA an toàn thiết bị (device-safe)
#  - Early-stopping theo Macro-F1 (VAL)
#  - Temperature scaling (VAL)
#  - Logit-bias search (VAL) để đẩy 7/10
#  - (Tuỳ chọn) Finetune head tách rời
#  - (Tuỳ chọn) OvR blend cho 7/10
# ============================
import math, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_recall_fscore_support

# ===== CỜ TUỲ CHỌN =====
ENABLE_HEAD_FINETUNE = True      # finetune riêng head vài epoch để nhạy với 7/10
ENABLE_OVR_BLEND     = False     # OvR cho 7/10 (LogReg) rồi blend vào xác suất DNN
OPT_BIAS_FOR_TARGETS = False     # True: tối ưu macro-F1 chỉ trên {7,10} khi chọn T & bias

# ===== Device & seed =====
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

# ===== Dữ liệu =====
assert "Label" in train.columns and "Label" in test.columns, "Thiếu cột Label trong train/test!"
X_full = train.drop(columns=["Label"]).values
y_full = train["Label"].astype(int).values
X_test = test.drop(columns=["Label"]).values
y_test = test["Label"].astype(int).values

num_classes  = int(max(y_full.max(), y_test.max())) + 1
num_features = X_full.shape[1]

# ===== 1) Stratified split: train_in / val (15%) =====
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(X_full, y_full))
X_tr_in, y_tr_in = X_full[train_idx], y_full[train_idx]
X_val,   y_val   = X_full[val_idx],   y_full[val_idx]

# ===== 2) Scaler =====
scaler = StandardScaler()
X_tr_in_sc = scaler.fit_transform(X_tr_in)
X_val_sc   = scaler.transform(X_val)


# ===== 3) Sampler cân bằng + boost 7/10 =====
counts = np.bincount(y_tr_in, minlength=num_classes).astype(np.float32)
inv = counts.sum() / (counts + 1e-9)
inv /= inv.mean()
boost = np.ones(num_classes, dtype=np.float32)
boost[[7,10]] = 1.6  # tăng tần suất gặp 7/10 (giữ hệ số)
sample_w = (inv * boost)[y_tr_in]
sampler  = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)

# ===== 4) DataLoaders =====
Xtr_t = torch.tensor(X_tr_in_sc, dtype=torch.float32)
ytr_t = torch.tensor(y_tr_in,    dtype=torch.long)
Xva_t = torch.tensor(X_val_sc,   dtype=torch.float32)
yva_t = torch.tensor(y_val,      dtype=torch.long)
Xte_t = torch.tensor(X_test_sc,  dtype=torch.float32)
yte_t = torch.tensor(y_test,     dtype=torch.long)

batch_size = 1024
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, sampler=sampler, pin_memory=True)
val_loader   = DataLoader(TensorDataset(Xva_t, yva_t), batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(TensorDataset(Xte_t, yte_t), batch_size=4096, shuffle=False, pin_memory=True)

# ===== 5) Mô hình =====
class ResidualBlock(nn.Module):
    def __init__(self, d_in, d_hid, p=0.25):
        super().__init__()
        self.lin1 = nn.Linear(d_in, d_hid)
        self.bn1  = nn.BatchNorm1d(d_hid)
        self.lin2 = nn.Linear(d_hid, d_in)
        self.ln2  = nn.LayerNorm(d_in)
        self.drop = nn.Dropout(p)
    def forward(self, x):
        h = self.drop(torch.relu(self.bn1(self.lin1(x))))
        h = self.lin2(h)
        return torch.relu(self.ln2(x + h))

class ResDNN(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        W = 512
        self.stem = nn.Sequential(nn.Linear(in_dim, W), nn.BatchNorm1d(W), nn.ReLU(), nn.Dropout(0.30))
        self.block1 = ResidualBlock(W, W//2, p=0.30)
        self.block2 = ResidualBlock(W, W//2, p=0.25)
        self.block3 = ResidualBlock(W, W//2, p=0.20)
        self.head   = nn.Linear(W, n_classes)
    def forward(self, x):
        h = self.stem(x)
        h = self.block1(h); h = self.block2(h); h = self.block3(h)
        return self.head(h)

model = ResDNN(num_features, num_classes).to(device)

# ===== 6) CB-Focal Loss + class weights (Effective Number) =====
def cb_class_weights(counts, beta=0.999):
    eff = (1.0 - beta) / (1.0 - np.power(beta, counts + 1e-12))
    w   = eff / eff.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

class FocalCE(nn.Module):
    def __init__(self, gamma=1.7, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
    def forward(self, logits, target):
        logp = torch.log_softmax(logits, dim=1)
        p    = torch.exp(logp)
        idx  = torch.arange(logits.size(0), device=logits.device)
        pt   = p[idx, target]
        loss = -((1-pt) ** self.gamma) * logp[idx, target]
        if self.weight is not None:
            loss = loss * self.weight[target]
        return loss.mean()

cls_w  = cb_class_weights(counts, beta=0.999)
criterion = FocalCE(gamma=1.7, weight=cls_w)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-4)

# ===== 7) Lịch LR: warmup + cosine =====
total_epochs  = 120
warmup_epochs = 8
steps_per_epoch = max(1, len(train_loader))
def lr_lambda(current_step):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs * steps_per_epoch
    if current_step < warmup_steps:
        return (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, (total_steps - warmup_steps))
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ===== 8) Mixup (giảm khi batch có 7/10) =====
def mixup_dynamic(x, y, base_alpha=0.10):
    alpha = base_alpha
    if any(lbl in (7,10) for lbl in y.tolist()):
        alpha = 0.02
    if alpha <= 0: return x, y, 1.0, None
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    x_m  = lam * x + (1 - lam) * x[perm]
    y_perm = y[perm]
    return x_m, y, lam, y_perm

# ===== 9) EMA (float-only) — DEVICE SAFE =====
ema_decay = 0.995
ema_state = {}
for k, v in model.state_dict().items():
    if v.is_floating_point():
        ema_state[k] = v.detach().clone().to(device=v.device, dtype=v.dtype)

def ema_update(model, ema_state, decay: float):
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if not v.is_floating_point():
                continue
            if (ema_state[k].device != v.device) or (ema_state[k].dtype != v.dtype):
                ema_state[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
            ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)

def load_ema_state(model, ema_state):
    current = model.state_dict()
    merged = {}
    for k, v in current.items():
        if k in ema_state:
            merged[k] = ema_state[k].to(device=v.device, dtype=v.dtype)
        else:
            merged[k] = v
    model.load_state_dict(merged, strict=True)

# ===== 10) Train + Early-stopping Macro-F1 (VAL) =====
best_f1, best_ema_snapshot, wait, patience = -1.0, None, 0, 20
global_step = 0

for epoch in range(total_epochs):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_m, ya, lam, yb_ = mixup_dynamic(xb, yb, base_alpha=0.10)
        logits = model(xb_m)
        loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_) if yb_ is not None else criterion(logits, ya)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        scheduler.step()
        ema_update(model, ema_state, ema_decay)
        global_step += 1

    # --- Eval VAL với EMA ---
    saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    load_ema_state(model, ema_state)
    model.eval()
    with torch.no_grad():
        logits_val = []
        for xb, _ in val_loader:
            logits_val.append(model(xb.to(device)).cpu())
        logits_val = torch.cat(logits_val, 0).numpy()
    pred_val = logits_val.argmax(1)
    f1_val_all = f1_score(y_val, pred_val, average='macro')
    f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
    f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all

    if f1_val > best_f1:
        best_f1, wait = f1_val, 0
        best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
    else:
        wait += 1

    model.load_state_dict(saved_live, strict=True)

    if wait >= patience:
        print(f"[Early-stop] epoch {epoch+1} | best VAL Macro-F1 ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}) = {best_f1:.4f}")
        break

# ===== 11) Load EMA tốt nhất và (tuỳ chọn) finetune head =====
if best_ema_snapshot is not None:
    ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
    load_ema_state(model, ema_state)

if ENABLE_HEAD_FINETUNE:
    for p in model.stem.parameters():   p.requires_grad = False
    for p in model.block1.parameters(): p.requires_grad = False
    for p in model.block2.parameters(): p.requires_grad = False
    for p in model.block3.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=5e-4, weight_decay=1e-4)
    ft_epochs, ft_patience, ft_wait, best_ft = 12, 6, 0, -1.0
    for epoch in range(ft_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)  # tắt mixup khi finetune head
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.head.parameters(), 3.0)
            optimizer.step()
            ema_update(model, ema_state, ema_decay)
        # eval VAL
        saved_live = {k: v.detach().clone() for k, v in model.state_dict().items()}
        load_ema_state(model, ema_state)
        model.eval()
        with torch.no_grad():
            logits_val = []
            for xb, _ in val_loader:
                logits_val.append(model(xb.to(device)).cpu())
            logits_val = torch.cat(logits_val, 0).numpy()
        pred_val = logits_val.argmax(1)
        f1_val_all = f1_score(y_val, pred_val, average='macro')
        f1_val_target = f1_score(y_val, pred_val, labels=[7,10], average='macro', zero_division=0)
        f1_val = f1_val_target if OPT_BIAS_FOR_TARGETS else f1_val_all
        if f1_val > best_ft:
            best_ft, ft_wait = f1_val, 0
            best_ema_snapshot = {k: v.detach().cpu().clone() for k, v in ema_state.items()}
        else:
            ft_wait += 1
        model.load_state_dict(saved_live, strict=True)
        if ft_wait >= ft_patience:
            print(f"[Finetune head Early-stop] epoch {epoch+1} | best VAL Macro-F1 = {best_ft:.4f}")
            break
    if best_ema_snapshot is not None:
        ema_state = {k: t.clone().to(device) for k, t in best_ema_snapshot.items()}
        load_ema_state(model, ema_state)

model.eval()

# ===== 12) Temperature scaling (VAL) =====
def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

with torch.no_grad():
    logits_val = []
    for xb, _ in val_loader:
        logits_val.append(model(xb.to(device)).cpu())
    logits_val = torch.cat(logits_val, 0).numpy()
    logits_test = []
    for xb, _ in test_loader:
        logits_test.append(model(xb.to(device)).cpu())
    logits_test = torch.cat(logits_test, 0).numpy()

best_T, best_val_f1 = 1.0, -1.0
for T in np.arange(0.5, 3.05, 0.1):
    p_val = softmax_np(logits_val / T)
    pred  = p_val.argmax(1)
    f1_all    = f1_score(y_val, pred, average='macro')
    f1_target = f1_score(y_val, pred, labels=[7,10], average='macro', zero_division=0)
    f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
    if f1_use > best_val_f1:
        best_val_f1, best_T = f1_use, float(T)

# ===== 13) (Tuỳ chọn) OvR blend cho 7/10 =====
if ENABLE_OVR_BLEND:
    from sklearn.linear_model import LogisticRegression
    X_ovr = X_tr_in_sc; y_ovr = y_tr_in
    ovr_targets = [7,10]
    lr_models = {}
    for t in ovr_targets:
        y_bin = (y_ovr == t).astype(int)
        lr = LogisticRegression(max_iter=200)
        lr.fit(X_ovr, y_bin)
        lr_models[t] = lr
    ovr_scores_test = {t: lr_models[t].predict_proba(X_test_sc)[:,1] for t in ovr_targets}

# ===== 14) Logit-bias search trên VAL để đẩy 7/10 =====
targets = [7,10]
grid = np.arange(0.0, 1.30, 0.10)
best_b, best_f1_bias = np.zeros(num_classes), -1.0

for b7 in grid:
    for b10 in grid:
        b = np.zeros(num_classes, dtype=np.float32)
        b[7], b[10] = b7, b10
        p_val = softmax_np((logits_val / best_T) + b[None, :])
        pred = p_val.argmax(1)
        f1_all    = f1_score(y_val, pred, average='macro')
        f1_target = f1_score(y_val, pred, labels=targets, average='macro', zero_division=0)
        f1_use    = f1_target if OPT_BIAS_FOR_TARGETS else f1_all
        if f1_use > best_f1_bias:
            best_f1_bias, best_b = f1_use, b.copy()

# ===== 15) Suy luận TEST với T* và bias* (và OvR nếu bật) =====
p_test = softmax_np((logits_test / best_T) + best_b[None, :])

if ENABLE_OVR_BLEND:
    for t in [7,10]:
        s = ovr_scores_test[t]  # [N]
        p_test[:, t] *= (0.5 + 0.5 * s)
    p_test = p_test / p_test.sum(axis=1, keepdims=True)

y_pred = p_test.argmax(1)

# ===== 16) Đánh giá =====
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes))
print(f"\nDNN (EMA, T={best_T:.2f}) — Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Best VAL-F1(T): {best_val_f1:.4f}")
print(f"Logit-bias best on VAL ({'targets' if OPT_BIAS_FOR_TARGETS else 'all'}): {best_f1_bias:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

labels_arr = np.arange(num_classes)
prec, rec, f1c, support = precision_recall_fscore_support(
    y_test, y_pred, labels=labels_arr, zero_division=0
)

# FPR per class
fpr_per_class = []
for i in labels_arr:
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    fpr_i = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    fpr_per_class.append(fpr_i)

macro_precision = float(np.mean(prec))
macro_recall    = float(np.mean(rec))
macro_fpr       = float(np.mean(fpr_per_class))

print("\n==== Macro metrics ====")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall/TPR: {macro_recall:.4f}")
print(f"Macro-FPR: {macro_fpr:.4f}")

per_class_df = pd.DataFrame({
    "class": labels_arr,
    "precision": prec,
    "recall_TPR": rec,
    "f1": f1c,
    "FPR": fpr_per_class,
    "support": support,
}).round(4)

print("\n==== Per-class metrics (all classes) ====")
print(per_class_df)

In [ ]:
# Save ResDNN checkpoint (compatible with ResDNNWrapper.from_checkpoint)
import os, torch

os.makedirs('./models', exist_ok=True)

checkpoint = {
    'state_dict': model.state_dict(),
    'scaler': scaler,
    'input_dim': num_features,
    'num_classes': num_classes,
}

save_path = './models/framework_resdnn_TVAE.pth'
torch.save(checkpoint, save_path)
print(f"ResDNN checkpoint saved to {save_path}")
print(f"  input_dim={num_features}, num_classes={num_classes}")